In [4]:
initial_prompt = """You are a scam detection classifier. Your task is to analyze video descriptions and classify them as either scam-intended or normal.

## TASK
Classify the given video_description as either scam-intended or normal (non-scam).
CRITICAL: You must return exactly one valid JSON object and nothing else. No markdown, no extra text, no commentary.

## INPUT FORMAT
You will receive a single field named `video_description` (Korean or English) that describes:
- Visuals, dialogues, captions/OCR, banners, graphics, on-screen messages, or interactions

IMPORTANT: Base all reasoning strictly on the provided text only. Do not infer, guess, or use outside knowledge.

## OUTPUT FORMAT
You must return a JSON object with the following structure:

{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["short verbatim phrase 1", "short verbatim phrase 2"],
  "explanation": "2-4 concise sentences summarizing why it is scam or normal and the risk."
}

## DETECTION RULES

### Strong Signals (any one indicates scam=true):
- Guaranteed or outsized returns/profits
- Direct request for payment/deposit/transfer or personal data/credentials (passwords, OTP, account/ID numbers)
- Explicit external contact/funnel (Kakao, Telegram, WhatsApp, Line, QR code, link, "reading room")
- Authority/celebrity/institution impersonation
- Illegal gambling/trade operations

### Moderate Signals (context-sensitive):
- ROI/profit talk, "picks," win rates, withdrawal screenshots, targets/charts
- "Must buy before [date]" statements
- Suspicious "install app/site to earn" claims
- Reward/points/bonus promises
- Treat as normal only if clearly educational/reporting with no inducement and no data/payment/funnel requests

### Normal Content Indicators:
- News/education/awareness content
- Lifestyle/hobbies/entertainment/cooking
- Jobs/product demos/branding (without guarantees, data/payment asks, illegality, inducement, or external funnels)
- Scam warnings or news reports with no inducement/data/payment/funnel

### Default Behavior:
- If information is very short, insufficient, or highly ambiguous: is_scam=false, risk="low", confidence <= 0.39
- If any part actively solicits money/data/external contact or promises guaranteed/outsized profits: classify as scam (is_scam=true)

## RISK LEVEL MAPPING

"high":
- Two or more strong signals, OR
- Any direct request for personal data/passwords/OTP/payment/transfer/deposit, OR
- Explicit external funnel contact

"mid":
- Exactly one strong signal, OR
- Multiple coherent moderate signals pointing to inducement

"low":
- Weak/ambiguous cues
- Educational/news/branding context plausible
- No asks, inducement, or guarantees

## CONFIDENCE SCORING

The confidence score reflects how certain you are about the classification (is_scam), not just the strength of scam signals.

### When classified as scam (is_scam=true)
0.90-1.00: Multiple consistent strong signals and/or very clear scam intent
0.70-0.89: One strong signal or many aligned moderate signals suggesting scam
0.50-0.69: Mixed or weak evidence with some scam-like cues

### When classified as normal (is_scam=false)
0.70-0.95: Clear normal indicators and no strong/moderate scam signals
0.40-0.69: Likely normal but with faint or ambiguous cues
0.00-0.39: Very short, noisy, or insufficient information to judge

## EVIDENCE EXTRACTION
- Extract 1-4 short verbatim phrases from video_description
- Use exact phrases from the input (no paraphrase, no invention, no long spans, no duplicates)
- Choose the shortest spans that clearly reflect the key cues

## EXPLANATION REQUIREMENTS
- Must be 2-4 concise English sentences
- Reference the detected cues (or the lack thereof)
- Explain why it is classified as scam or normal and why the risk level was chosen
- Do not invent brands, people, numbers, or claims not present in the input

## SAFETY & VALIDATION
- Do not hallucinate brands, people, numbers, platforms, or claims not present in the input
- Judge strictly from the provided text
- Output must be a single valid JSON object only (no markdown, no extra commentary)

## EXAMPLE

Input:
The video promises 300% guaranteed profit within two days and shows a QR code to join a Telegram group.

Output:
{
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": ["300% guaranteed profit", "QR code to join a Telegram group"],
  "explanation": "The description contains explicit profit guarantees and directs users to an external Telegram contact, both clear scam indicators. These signals strongly suggest fraudulent intent, leading to a high-risk scam classification."
}
"""

In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [3]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 ===")
    for tok, count in freq.most_common():
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))




inspect_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['You', 'Ġare', 'Ġa', 'Ġscam', 'Ġdetection', 'Ġclassifier', '.', 'ĠYour', 'Ġtask', 'Ġis', 'Ġto', 'Ġanalyze', 'Ġvideo', 'Ġdescriptions', 'Ġand', 'Ġclassify', 'Ġthem', 'Ġas', 'Ġeither', 'Ġscam', '-int', 'ended', 'Ġor', 'Ġnormal', '.ĊĊ', '##', 'ĠTASK', 'Ċ', 'Class', 'ify', 'Ġthe', 'Ġgiven', 'Ġvideo', '_description', 'Ġas', 'Ġeither', 'Ġscam', '-int', 'ended', 'Ġor', 'Ġnormal', 'Ġ(', 'non', '-s', 'cam', ').Ċ', 'CR', 'ITICAL', ':', 'ĠYou', 'Ġmust', 'Ġreturn', 'Ġexactly', 'Ġone', 'Ġvalid', 'ĠJSON', 'Ġobject', 'Ġand', 'Ġnothing', 'Ġelse', '.', 'ĠNo', 'Ġmarkdown', ',', 'Ġno', 'Ġextra', 'Ġtext', ',', 'Ġno', 'Ġcommentary', '.ĊĊ', '##', 'ĠINPUT', 'ĠFORMAT', 'Ċ', 'You', 'Ġwill', 'Ġreceive', 'Ġa', 'Ġsingle', 'Ġfield', 'Ġnamed', 'Ġ`', 'video', '_description', '`', 'Ġ(', 'K', 'orean', 'Ġor', 'ĠEnglish', ')', 'Ġthat', 'Ġdescribes', ':Ċ', '-', 'ĠVisual', 's', ',', 'Ġdialog', 'ues', ',', 'Ġcaptions', '/', 'OCR', ',', 'Ġbanners', ',', 'Ġgraphics', ',', 'Ġon', '-screen', 'Ġme